# Overview

This notebook presents a **fusion-based alternative** to the pure Vision Transformer (ViT) for breast-cancer image classification on a subset of the RSNA Breast Cancer Detection dataset.

Instead of relying on a single ViT, we build a **dual-branch CNN–Transformer fusion** network:

* a **convolutional branch** that captures *local* texture / edge detail (strong inductive bias, data-efficient), and
* a **transformer branch** that captures *global* long-range context (the original ViT path),

and we merge them with a **cross-attention fusion** block. The dataset, pre-processing and the Ignite training loop are kept identical to the original ViT notebook, so this model is a drop-in alternate you can compare directly.

# 1. Why a fusion architecture?

A pure ViT treats an image as a flat sequence of patches. It has very little built-in prior about *locality* (neighbouring pixels are related) or *translation equivariance*, so it is powerful but data-hungry — a real concern for medical datasets that are small and imbalanced.

A CNN is the opposite: convolutions bake in locality and translation equivariance, making them data-efficient at modelling fine local structure (micro-calcifications, mass margins), but their receptive field grows slowly, so global context is harder to reach.

**Fusion** keeps the best of both. The idea is a two-stream (two-branch) model:

| Branch | Role | Output |
|---|---|---|
| CNN branch | Local features (texture, edges, calcifications) | A grid of local tokens |
| Transformer branch | Global features (long-range context) | Patch tokens + a `[CLS]` token |
| Fusion module | Let the global stream *attend to* the local stream | A fused `[CLS]` representation |

There are three common ways to fuse the two streams:

1. **Late / concat fusion** — pool each branch to a vector and concatenate, then classify. Simple, but the branches never *talk* to each other.
2. **Cross-attention fusion** — use one stream as the *query* and the other as *key/value*, so every global token can pull in the local detail it needs. This is what we implement below.
3. **Interleaved fusion** (e.g. Conformer / CMT) — exchange features at every block. Most powerful, most expensive.

We implement **(2) cross-attention fusion** as the main model and keep **(1) concat fusion** as a lightweight variant for comparison.

![two-branch fusion](https://production-media.paperswithcode.com/methods/Screen_Shot_2021-01-26_at_9.43.31_PM_uI4jjMq.png)
*(The transformer branch is the classic ViT above; the CNN branch runs alongside it and is merged by cross-attention.)*

# 2. About the Dataset

Same as the original notebook. The data was contributed by mammography screening programs in Australia and the U.S., with radiologist evaluations and follow-up pathology for suspected malignancies. We use an already-cropped, 512×512 PNG version of the RSNA dataset so we can skip DICOM decoding during training.

> If you work from raw DICOMs instead, use the `read_xray` helper below — naively rescaling DICOM to PNG/JPG harms the data; see [this notebook](https://www.kaggle.com/code/raddar/convert-dicom-to-np-array-the-correct-way/notebook).

# 3. Implementation

## 3.1 Utility functions

These are shared with the original ViT notebook: a DICOM reader, a `patchify` op, and a small MLP factory.

In [ ]:
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from ignite.engine import Events, create_supervised_trainer, create_supervised_evaluator
from ignite.metrics import Accuracy, Loss, RunningAverage
from ignite.contrib.handlers import ProgressBar
from sklearn.model_selection import train_test_split
from torchvision import models, transforms

In [ ]:
def read_xray(file_path, img_size=None):
    """
    Read a DICOM file and return a normalised single-channel image.
    """
    import pydicom
    dicom = pydicom.read_file(file_path)
    img = dicom.pixel_array

    if dicom.PhotometricInterpretation == "MONOCHROME1":
        img = np.max(img) - img

    if img_size:
        img = cv2.resize(img, img_size)

    img = img[np.newaxis]                 # add channel dim at front
    img = img / np.max(img)               # scale to [0, 1]
    img = img.astype("float32")
    return img

In [ ]:
def patchify(batch, patch_size):
    """
    Split a batch of images into a grid of patches.

    Shape:
        batch:  (b, c, h, w)
        output: (b, c, nh, nw, ph, pw)
    """
    b, c, h, w = batch.shape
    ph, pw = patch_size
    nh, nw = h // ph, w // pw

    batch_patches = torch.reshape(batch, (b, c, nh, ph, nw, pw))
    batch_patches = torch.permute(batch_patches, (0, 1, 2, 4, 3, 5))
    return batch_patches

In [ ]:
def get_mlp(in_features, hidden_units, out_features):
    """Return an MLP head: Linear -> ReLU -> ... -> Linear."""
    dims = [in_features] + hidden_units + [out_features]
    layers = []
    for dim1, dim2 in zip(dims[:-2], dims[1:-1]):
        layers.append(nn.Linear(dim1, dim2))
        layers.append(nn.ReLU())
    layers.append(nn.Linear(dims[-2], dims[-1]))
    return nn.Sequential(*layers)

## 3.2 Transformer branch — Image to Sequence

Identical to the original ViT: turn an image into a sequence of linearly-projected patch tokens, add a learnable positional embedding, and prepend a `[CLS]` token.

In [ ]:
class Img2Seq(nn.Module):
    """
    Turn a batch of images into a batch of token sequences.

    Shape:
        input:  (b, c, h, w)
        output: (b, s + 1, d)   # +1 for the [CLS] token
    """
    def __init__(self, img_size, patch_size, n_channels, d_model):
        super().__init__()
        self.patch_size = patch_size
        self.img_size = img_size

        nh, nw = img_size[0] // patch_size[0], img_size[1] // patch_size[1]
        n_tokens = nh * nw

        token_dim = patch_size[0] * patch_size[1] * n_channels
        self.linear = nn.Linear(token_dim, d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.pos_emb = nn.Parameter(torch.randn(n_tokens, d_model))

    def forward(self, batch):
        batch = patchify(batch, self.patch_size)
        b, c, nh, nw, ph, pw = batch.shape

        # flatten each patch
        batch = torch.permute(batch, [0, 2, 3, 4, 5, 1])
        batch = torch.reshape(batch, [b, nh * nw, ph * pw * c])

        batch = self.linear(batch)
        cls = self.cls_token.expand([b, -1, -1])
        emb = batch + self.pos_emb
        return torch.cat([cls, emb], axis=1)

## 3.3 CNN branch — Image to local tokens

The convolutional branch is a small stack of strided `Conv → BatchNorm → ReLU` blocks. Each stride-2 block halves the spatial resolution, so with four blocks a 512×512 image becomes a `32×32` feature map with `d_model` channels.

We then flatten the spatial grid into a sequence of **`32 × 32 = 1024` local tokens** of dimension `d_model`. Note this is exactly the same number of tokens as the ViT patch grid (`512 / 16 = 32`), which makes the two branches easy to align and fuse.

In [ ]:
class CNNBranch(nn.Module):
    """
    A lightweight convolutional stem producing a sequence of local tokens.

    Shape:
        input:  (b, c, H, W)
        output: (b, s_cnn, d)      # s_cnn = (H/16) * (W/16)
    """
    def __init__(self, n_channels, d_model):
        super().__init__()

        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, kernel_size=3, stride=2, padding=1),
                nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True),
            )

        self.features = nn.Sequential(
            block(n_channels, 32),     # 512 -> 256
            block(32, 64),             # 256 -> 128
            block(64, 128),            # 128 -> 64
            block(128, d_model),       # 64  -> 32
        )

    def forward(self, x):
        x = self.features(x)               # (b, d_model, 32, 32)
        b, c, h, w = x.shape
        x = x.flatten(2).transpose(1, 2)   # (b, 32*32, d_model) = (b, 1024, d_model)
        return x

## 3.4 Fusion module — Cross-attention

This is the heart of the architecture. We let the **transformer tokens be the queries** and the **CNN tokens be the keys and values**. So every global/semantic token can look across the whole image and *pull in* the local convolutional detail that is most relevant to it.

It is a standard transformer-decoder-style block, but with the key/value stream coming from a *different* branch:

```
fused = LayerNorm( vit_tokens + CrossAttention(Q=vit, K=cnn, V=cnn) )
fused = LayerNorm( fused      + FeedForward(fused) )
```

The output keeps the ViT token layout (including the `[CLS]` token at index 0), so downstream we simply read the fused `[CLS]` token for classification.

In [ ]:
class CrossAttentionFusion(nn.Module):
    """
    Fuse two token streams with cross-attention.
    Query  = transformer tokens (global)
    Key/Value = CNN tokens (local)

    Shape:
        vit_tokens: (b, s_vit, d)
        cnn_tokens: (b, s_cnn, d)
        output:     (b, s_vit, d)
    """
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.1):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(
            d_model, nhead, dropout=dropout, batch_first=True
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, vit_tokens, cnn_tokens):
        attn_out, _ = self.cross_attn(vit_tokens, cnn_tokens, cnn_tokens)
        x = self.norm1(vit_tokens + self.dropout(attn_out))
        x = self.norm2(x + self.ffn(x))
        return x

## 3.5 The Fusion model

We wire the three pieces together:

1. **Transformer branch** — `Img2Seq` → `TransformerEncoder` (the original ViT path).
2. **CNN branch** — `CNNBranch` → a grid of local tokens.
3. **Cross-attention fusion** — one (or more) fusion blocks, then an MLP head on the fused `[CLS]` token.

The constructor signature matches the original `ViT` so you can swap models by changing one class name.

In [ ]:
class FusionViT(nn.Module):
    def __init__(
        self,
        img_size,
        patch_size,
        n_channels,
        d_model,
        nhead,
        dim_feedforward,
        blocks,
        mlp_head_units,
        n_classes,
        fusion_blocks=1,
    ):
        super().__init__()
        """
        Args:
            img_size:        Size of the image, e.g. (512, 512)
            patch_size:      Size of a ViT patch, e.g. (16, 16)
            n_channels:      Number of image channels
            d_model:         Feature width shared by both branches
            nhead:           Heads in every multi-head attention
            dim_feedforward: Hidden width of the feed-forward sub-layers
            blocks:          Number of transformer-encoder layers (ViT branch)
            mlp_head_units:  Hidden units of the classification MLP
            n_classes:       Number of output classes
            fusion_blocks:   Number of stacked cross-attention fusion layers
        """
        # --- Transformer (global) branch ---
        self.img2seq = Img2Seq(img_size, patch_size, n_channels, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model, nhead, dim_feedforward, activation="gelu", batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, blocks)

        # --- Convolutional (local) branch ---
        self.cnn_branch = CNNBranch(n_channels, d_model)

        # --- Cross-attention fusion ---
        self.fusion = nn.ModuleList([
            CrossAttentionFusion(d_model, nhead, dim_feedforward)
            for _ in range(fusion_blocks)
        ])

        # --- Classification head ---
        self.mlp = get_mlp(d_model, mlp_head_units, n_classes)
        self.output = nn.Sigmoid() if n_classes == 1 else nn.Softmax(dim=-1)

    def forward(self, batch):
        # global tokens: (b, s_vit + 1, d)
        vit_tokens = self.transformer_encoder(self.img2seq(batch))
        # local tokens: (b, s_cnn, d)
        cnn_tokens = self.cnn_branch(batch)

        # fuse: global queries attend to local keys/values
        fused = vit_tokens
        for layer in self.fusion:
            fused = layer(fused, cnn_tokens)

        cls = fused[:, 0, :]              # fused [CLS] token
        return self.output(self.mlp(cls))

### Sanity check — shapes

A quick forward pass on random data confirms the branches align and the model returns one probability per image.

In [ ]:
_device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

_m = FusionViT(
    img_size=(512, 512), patch_size=(16, 16), n_channels=1,
    d_model=256, nhead=4, dim_feedforward=512, blocks=4,
    mlp_head_units=[256, 128], n_classes=1, fusion_blocks=1,
).to(_device)

_x = torch.randn(2, 1, 512, 512, device=_device)
with torch.no_grad():
    _y = _m(_x)

n_params = sum(p.numel() for p in _m.parameters()) / 1e6
print("output shape:", tuple(_y.shape))          # (2, 1)
print(f"parameters:   {n_params:.2f} M")
del _m, _x, _y

### (Optional) Concat-fusion variant

For comparison, here is the simpler **late / concat fusion**: pool each branch to a single vector and concatenate before the head. The branches never attend to each other, so it is cheaper but usually a little weaker.

In [ ]:
class ConcatFusionViT(nn.Module):
    """Late-fusion baseline: mean-pool each branch, concat, then classify."""
    def __init__(self, img_size, patch_size, n_channels, d_model, nhead,
                 dim_feedforward, blocks, mlp_head_units, n_classes):
        super().__init__()
        self.img2seq = Img2Seq(img_size, patch_size, n_channels, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model, nhead, dim_feedforward, activation="gelu", batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, blocks)
        self.cnn_branch = CNNBranch(n_channels, d_model)

        self.mlp = get_mlp(2 * d_model, mlp_head_units, n_classes)
        self.output = nn.Sigmoid() if n_classes == 1 else nn.Softmax(dim=-1)

    def forward(self, batch):
        vit_cls = self.transformer_encoder(self.img2seq(batch))[:, 0, :]  # (b, d)
        cnn_vec = self.cnn_branch(batch).mean(dim=1)                      # (b, d)
        fused = torch.cat([vit_cls, cnn_vec], dim=1)                      # (b, 2d)
        return self.output(self.mlp(fused))

# 4. Training

A simple Ignite training loop — identical in structure to the original ViT notebook, so results are directly comparable.

## 4.1 Model Hyperparameters

In [ ]:
img_size = (512, 512)
patch_size = (16, 16)
n_channels = 1
d_model = 256            # shared width of both branches (lighter than the 1024 pure-ViT)
nhead = 4
dim_feedforward = 512
blocks = 4              # transformer-encoder layers in the ViT branch
mlp_head_units = [256, 128]
n_classes = 1
fusion_blocks = 1       # number of cross-attention fusion layers

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("device:", device)

> **Note on memory.** A two-branch model is heavier than a single ViT, so we use a smaller `d_model` (256 vs 1024). The CNN branch down-samples with strided convs (not a huge dense ViT over 1024-dim tokens), so the extra cost is modest. Lower the batch size if you hit OOM on your GPU.

## 4.2 Dataset and DataLoader

In [ ]:
class RSNADataset(Dataset):
    def __init__(self, df, img_path):
        self.df = df
        self.img_path = img_path

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        patient_id, image_id, cancer = self.df.iloc[idx][["patient_id", "image_id", "cancer"]]
        file = os.path.join(self.img_path, f"{patient_id}_{image_id}.png")
        file = cv2.imread(file, cv2.COLOR_BGR2GRAY)

        # CLAHE contrast enhancement
        clahe = cv2.createCLAHE(clipLimit=15, tileGridSize=[8, 8])
        file = clahe.apply(file)

        file = file / file.max()
        X = torch.tensor(file[np.newaxis].astype("float32")).to(device)
        y = torch.tensor([cancer]).float().to(device)
        return X, y

In [ ]:
df = pd.read_csv("/kaggle/input/rsna-breast-cancer-detection/train.csv")
counts = df["cancer"].value_counts()
df["weights"] = df["cancer"].apply(lambda x: 1 / counts[x])

train_df, val_df = train_test_split(df, test_size=0.25, stratify=df["cancer"])

In [ ]:
img_path = "/kaggle/input/rsna-breast-cancer-512-pngs"
train_samples = 1000
val_samples = 500

train_ds = RSNADataset(train_df, img_path)
val_ds = RSNADataset(val_df, img_path)

# WeightedRandomSampler handles the heavy class imbalance
train_sampler = WeightedRandomSampler(train_df["weights"].values, train_samples)
train_loader = DataLoader(train_ds, batch_size=8, sampler=train_sampler)

val_sampler = WeightedRandomSampler(val_df["weights"].values, val_samples)
val_loader = DataLoader(val_ds, batch_size=32, sampler=val_sampler)

## 4.3 Train the Fusion model

In [ ]:
model = FusionViT(
    img_size=img_size,
    patch_size=patch_size,
    n_channels=n_channels,
    d_model=d_model,
    nhead=nhead,
    dim_feedforward=dim_feedforward,
    blocks=blocks,
    mlp_head_units=mlp_head_units,
    n_classes=n_classes,
    fusion_blocks=fusion_blocks,
).to(device)

optimizer = Adam(model.parameters())
criterion = nn.BCELoss()

trainer = create_supervised_trainer(model, optimizer, criterion, device=device)
val_metrics = {"bce": Loss(criterion)}
evaluator = create_supervised_evaluator(model, metrics=val_metrics, device=device)

In [ ]:
log_interval = 10
max_epochs = 5
best_loss = float("inf")

RunningAverage(output_transform=lambda x: x).attach(trainer, "loss")

pbar = ProgressBar()
pbar.attach(trainer, ["loss"])


@trainer.on(Events.EPOCH_COMPLETED)
def log_validation_results(trainer):
    global best_loss
    evaluator.run(val_loader)
    loss = evaluator.state.metrics["bce"]
    if loss < best_loss:
        best_loss = loss
        torch.save(model.state_dict(), "best_model_fusion.pt")
    print(f"Validation Results - Epoch: {trainer.state.epoch} Avg loss: {loss:.2f}")


output_state = trainer.run(train_loader, max_epochs=max_epochs)

## 4.4 Comparison with ResNet50

The same transfer-learning baseline as the original notebook, trained on the same loaders.

In [ ]:
resnet = models.resnet50(pretrained=True)
in_features = resnet.fc.in_features
resnet.fc = nn.Linear(in_features, 1)

resnet_transforms = transforms.Compose([
    transforms.Resize((228, 228)),
    transforms.Lambda(lambda x: x.repeat([1, 3, 1, 1])),   # 1-ch -> 3-ch
])


class MyResNet(nn.Module):
    def __init__(self, transforms, model):
        super().__init__()
        self.transforms = transforms
        self.model = model
        self.output = nn.Sigmoid()

    def forward(self, batch):
        batch = self.transforms(batch)
        batch = self.model(batch)
        return self.output(batch)

In [ ]:
model = MyResNet(resnet_transforms, resnet).to(device)

optimizer = Adam(model.parameters())
criterion = nn.BCELoss()

trainer = create_supervised_trainer(model, optimizer, criterion, device=device)
val_metrics = {"bce": Loss(criterion)}
evaluator = create_supervised_evaluator(model, metrics=val_metrics, device=device)

In [ ]:
best_loss = float("inf")

RunningAverage(output_transform=lambda x: x).attach(trainer, "loss")

pbar = ProgressBar()
pbar.attach(trainer, ["loss"])


@trainer.on(Events.EPOCH_COMPLETED)
def log_validation_results(trainer):
    global best_loss
    evaluator.run(val_loader)
    loss = evaluator.state.metrics["bce"]
    if loss < best_loss:
        best_loss = loss
        torch.save(model.state_dict(), "best_model_resnet.pt")
    print(f"Validation Results - Epoch: {trainer.state.epoch} Avg loss: {loss:.2f}")


output_state = trainer.run(train_loader, max_epochs=max_epochs)

# 5. Recap & next steps

**What changed vs. the pure ViT.** We added a convolutional branch for local detail and a **cross-attention fusion** block so the global transformer stream can query the local CNN stream. This gives the model a CNN's data-efficiency *and* the transformer's global context — often a better fit for small, imbalanced medical datasets than either one alone.

**Ideas to push further:**

* **More fusion depth** — raise `fusion_blocks`, or make fusion *bidirectional* (also let CNN tokens attend to ViT tokens).
* **Interleaved fusion** — exchange features at every block (Conformer / CMT style) instead of once at the end.
* **Multimodal fusion** — the RSNA `train.csv` has rich tabular fields (age, view, laterality, implant, machine id). Encode them with a small MLP and fuse them with the image `[CLS]` token for true multi-modal fusion.
* **Better objective** — swap `BCELoss` for `BCEWithLogitsLoss` (drop the final `Sigmoid`) for numerical stability, add `pos_weight` for imbalance, and track **pF1 / AUROC** (the competition metric) rather than loss alone.
* **Pretrained CNN branch** — replace `CNNBranch` with a pretrained backbone (e.g. an EfficientNet/ResNet stem) for a stronger local stream.

If this notebook helped, consider trying the concat-fusion variant (`ConcatFusionViT`) as a cheaper baseline to confirm the cross-attention is pulling its weight.